In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Si, SEPD

This example demonstrates a Rietveld refinement of Si crystal
structure using time-of-flight neutron powder diffraction data from
SEPD at Argonne.

It also shows how to switch calculation engine and peak profile type.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.coord_system_code = '2'

### Set Unit Cell

In [5]:
structure.cell.length_a = 5.431

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.125,
    fract_y=0.125,
    fract_z=0.125,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their
parameters, and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-si-sepd', destination='data')

Getting data...


Data 'meas-si-sepd': Si, SEPD (Argonne)


✅ Data 'meas-si-sepd' already present at '../../../data/meas-si-sepd.xye'. Keeping existing.


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(
    name='sepd', data_path=data_path, beam_mode='time-of-flight'
)

### Set Instrument

In [9]:
expt.instrument.setup_twotheta_bank = 144.845
expt.instrument.calib_d_to_tof_offset = 0.0
expt.instrument.calib_d_to_tof_linear = 7476.91
expt.instrument.calib_d_to_tof_quadratic = -1.54

### Set Peak Profile

In [10]:
expt.peak.show_supported()
expt.peak.broad_gauss_sigma_0 = 3.0
expt.peak.broad_gauss_sigma_1 = 40.0
expt.peak.broad_gauss_sigma_2 = 2.0
expt.peak.decay_beta_0 = 0.04221
expt.peak.decay_beta_1 = 0.00946
expt.peak.rise_alpha_0 = 0.0
expt.peak.rise_alpha_1 = 0.5971

Peak types


,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


### Set Background

In [11]:
expt.background.type = 'line-segment'
for x in range(0, 35000, 5000):
    expt.background.create(id=str(x), position=x, intensity=200)

Background type for experiment 'sepd' already set to


line-segment


### Set Linked Structures

In [12]:
expt.linked_structures.create(structure_id='si', scale=10.0)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [13]:
project = Project(name='si_sepd')

### Add Structure

In [14]:
project.structures.add(structure)

### Add Experiment

In [15]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Display Structure

In [16]:
project.display.structure(struct_name='si')

Structure 🧩 'si' (Atom view type: 'covalent')


### Display Pattern

In [17]:
project.display.pattern(expt_name='sepd')
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 1/5

Set parameters to be refined.

In [18]:
structure.cell.length_a.free = True

expt.linked_structures['si'].scale.free = True
expt.instrument.calib_d_to_tof_offset.free = True

Show free parameters after selection.

In [19]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43100,,-inf,inf,Å
2,sepd,linked_structure,si,scale,10.00000,,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,0.00000,,-inf,inf,μs


#### Run Fitting

In [20]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),19.67
4,🔁 Iterations,27
5,📏 Goodness-of-fit (reduced χ²),66.72
6,"📏 R-factor (Rf, %)",23.08
7,"📏 R-factor squared (Rf², %)",12.55
8,"📏 Weighted R-factor (wR, %)",12.51


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4310,5.4314,0.0002,0.01 % ↑
2,sepd,linked_structure,si,scale,,10.0000,13.3619,0.1153,33.62 % ↑
3,sepd,instrument,,d_to_tof_offset,μs,0.0000,-9.2543,0.2503,N/A


#### Display Pattern

In [21]:
project.display.pattern(expt_name='sepd')

In [22]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 2/5

Set more parameters to be refined.

In [23]:
for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [24]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43135,0.00018,-inf,inf,Å
2,sepd,linked_structure,si,scale,13.36187,0.11531,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-9.25432,0.25033,-inf,inf,μs
4,sepd,background,0,intensity,200.00000,,-inf,inf,
5,sepd,background,5000,intensity,200.00000,,-inf,inf,
6,sepd,background,10000,intensity,200.00000,,-inf,inf,
7,sepd,background,15000,intensity,200.00000,,-inf,inf,
8,sepd,background,20000,intensity,200.00000,,-inf,inf,
9,sepd,background,25000,intensity,200.00000,,-inf,inf,
10,sepd,background,30000,intensity,200.00000,,-inf,inf,


#### Run Fitting

In [25]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),31.47
4,🔁 Iterations,45
5,📏 Goodness-of-fit (reduced χ²),3.38
6,"📏 R-factor (Rf, %)",9.29
7,"📏 R-factor squared (Rf², %)",6.33
8,"📏 Weighted R-factor (wR, %)",5.95


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4314,5.4314,0.0000,0.00 % ↑
2,sepd,linked_structure,si,scale,,13.3619,14.6317,0.0265,9.50 % ↑
3,sepd,instrument,,d_to_tof_offset,μs,-9.2543,-9.2534,0.0515,0.01 % ↓
4,sepd,background,0,intensity,,200.0000,268.6002,0.9745,34.30 % ↑
5,sepd,background,5000,intensity,,200.0000,144.7589,0.4071,27.62 % ↓
6,sepd,background,10000,intensity,,200.0000,120.0247,0.4282,39.99 % ↓
7,sepd,background,15000,intensity,,200.0000,135.8494,0.8169,32.08 % ↓
8,sepd,background,20000,intensity,,200.0000,132.6887,1.4317,33.66 % ↓
9,sepd,background,25000,intensity,,200.0000,175.1775,2.8755,12.41 % ↓
10,sepd,background,30000,intensity,,200.0000,180.4556,5.8525,9.77 % ↓


#### Display Pattern

In [26]:
project.display.pattern(expt_name='sepd')

In [27]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 3/5

Fix background points.

In [28]:
for point in expt.background:
    point.intensity.free = False

Set more parameters to be refined.

In [29]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_gauss_sigma_2.free = True

Show free parameters after selection.

In [30]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43137,0.00004,-inf,inf,Å
2,sepd,linked_structure,si,scale,14.63167,0.02651,-inf,inf,
3,sepd,peak,,broad_gauss_sigma_0,3.00000,,-inf,inf,μs²
4,sepd,peak,,broad_gauss_sigma_1,40.00000,,-inf,inf,μs/Å
5,sepd,peak,,broad_gauss_sigma_2,2.00000,,-inf,inf,μs²/Å²
6,sepd,instrument,,d_to_tof_offset,-9.25336,0.05153,-inf,inf,μs


#### Run Fitting

In [31]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),25.47
4,🔁 Iterations,36
5,📏 Goodness-of-fit (reduced χ²),3.21
6,"📏 R-factor (Rf, %)",8.99
7,"📏 R-factor squared (Rf², %)",5.52
8,"📏 Weighted R-factor (wR, %)",4.88


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4314,5.4314,0.0000,0.00 % ↑
2,sepd,linked_structure,si,scale,,14.6317,14.7057,0.0257,0.51 % ↑
3,sepd,peak,,broad_gauss_sigma_0,μs²,3.0000,5.7727,0.4206,92.42 % ↑
4,sepd,peak,,broad_gauss_sigma_1,μs/Å,40.0000,44.2827,0.7966,10.71 % ↑
5,sepd,peak,,broad_gauss_sigma_2,μs²/Å²,2.0000,1.2962,0.1680,35.19 % ↓
6,sepd,instrument,,d_to_tof_offset,μs,-9.2534,-9.2506,0.0546,0.03 % ↓


#### Display Pattern

In [32]:
project.display.pattern(expt_name='sepd')

In [33]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 4/5

Set more parameters to be refined.

In [34]:
structure.atom_sites['Si'].adp_iso.free = True

expt.peak.decay_beta_0.free = True
expt.peak.decay_beta_1.free = True
expt.peak.rise_alpha_1.free = True

Show free parameters after selection.

In [35]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43143,0.00004,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.50000,,-inf,inf,Å²
3,sepd,linked_structure,si,scale,14.70568,0.02567,-inf,inf,
4,sepd,peak,,rise_alpha_1,0.59710,,-inf,inf,μs/Å
5,sepd,peak,,decay_beta_0,0.04221,,-inf,inf,μs
6,sepd,peak,,decay_beta_1,0.00946,,-inf,inf,μs/Å
7,sepd,peak,,broad_gauss_sigma_0,5.77274,0.42055,-inf,inf,μs²
8,sepd,peak,,broad_gauss_sigma_1,44.28265,0.79664,-inf,inf,μs/Å
9,sepd,peak,,broad_gauss_sigma_2,1.29621,0.16795,-inf,inf,μs²/Å²
10,sepd,instrument,,d_to_tof_offset,-9.25065,0.05458,-inf,inf,μs


#### Run Fitting

In [36]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),69.15
4,🔁 Iterations,102
5,📏 Goodness-of-fit (reduced χ²),2.93
6,"📏 R-factor (Rf, %)",8.39
7,"📏 R-factor squared (Rf², %)",4.16
8,"📏 Weighted R-factor (wR, %)",2.48


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4314,5.4325,0.0001,0.02 % ↑
2,si,atom_site,Si,adp_iso,Å²,0.5000,0.5240,0.0033,4.80 % ↑
3,sepd,linked_structure,si,scale,,14.7057,14.9693,0.0355,1.79 % ↑
4,sepd,peak,,rise_alpha_1,μs/Å,0.5971,0.2370,0.0043,60.31 % ↓
5,sepd,peak,,decay_beta_0,μs,0.0422,0.0386,0.0002,8.65 % ↓
6,sepd,peak,,decay_beta_1,μs/Å,0.0095,0.0106,0.0002,11.67 % ↑
7,sepd,peak,,broad_gauss_sigma_0,μs²,5.7727,6.9657,0.4577,20.67 % ↑
8,sepd,peak,,broad_gauss_sigma_1,μs/Å,44.2827,25.6509,1.0250,42.07 % ↓
9,sepd,peak,,broad_gauss_sigma_2,μs²/Å²,1.2962,1.1001,0.1584,15.13 % ↓
10,sepd,instrument,,d_to_tof_offset,μs,-9.2506,-8.7248,0.0740,5.68 % ↓


#### Display Correlations

In [37]:
project.display.fit.correlations()

#### Display Pattern

In [38]:
project.display.pattern(expt_name='sepd')

In [39]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

In [40]:
project.display.pattern(expt_name='sepd', x='d_spacing')

### Perform Fit 5/5

#### Switch calculator engine

In [41]:
expt.calculator.show_supported()

Calculator types


,,Type,Description
1,,crysfml,CrysFML library for crystallographic calculations
2,*,cryspy,CrysPy library for crystallographic calculations


In [42]:
expt.calculator.type = 'crysfml'

Calculator for experiment 'sepd' changed to


crysfml


#### Change peak profile type

In [43]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
2,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt


In [44]:
expt.peak.type = 'jorgensen-von-dreele'

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • broad_lorentz_gamma_0=0.0                                                                                                    
   • broad_lorentz_gamma_1=0.0                                                                                                    
   • broad_lorentz_gamma_2=0.0                                                                                                    


⚠️ Switching peak profile type resets these settings to defaults:                                                                 
   • broad_gauss_sigma_0: 6.965738176548829 -> 7.0                                                                                
   • broad_gauss_sigma_1: 25.65086980218297 -> 0.0                                                                                
   • broad_gauss_sigma_2: 1.100094567751237 -> 0.0                                                                                
   • decay_beta_0: 0.03855828518533446 -> 0.04                                                                                    
   • decay_beta_1: 0.010563672457364238 -> 0.0                                                                                    
   • rise_alpha_1: 0.23701152844923368 -> 0.2                                                                                     


Peak profile type for experiment 'sepd' changed to


jorgensen-von-dreele


In [45]:
expt.peak.broad_gauss_sigma_0 = 3.0148
expt.peak.broad_gauss_sigma_1 = 33.3451
expt.peak.broad_lorentz_gamma_1 = 2.5489
expt.peak.decay_beta_0 = 0.04221
expt.peak.decay_beta_1 = 0.00946
expt.peak.rise_alpha_1 = 0.5971

#### Add new free parameters

In [46]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_lorentz_gamma_1.free = True
expt.peak.decay_beta_0.free = True
expt.peak.decay_beta_1.free = True
expt.peak.rise_alpha_1.free = True

#### Run Fitting

In [47]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),161.51
4,🔁 Iterations,398
5,📏 Goodness-of-fit (reduced χ²),2.67
6,"📏 R-factor (Rf, %)",7.97
7,"📏 R-factor squared (Rf², %)",4.08
8,"📏 Weighted R-factor (wR, %)",2.85


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4325,5.4310,N/A,0.03 % ↓
2,si,atom_site,Si,adp_iso,Å²,0.5240,0.5253,N/A,0.24 % ↑
3,sepd,linked_structure,si,scale,,14.9693,1134.0571,N/A,7475.90 % ↑
4,sepd,peak,,rise_alpha_1,μs/Å,0.5971,0.6491,N/A,8.70 % ↑
5,sepd,peak,,decay_beta_0,μs,0.0422,0.0411,N/A,2.52 % ↓
6,sepd,peak,,decay_beta_1,μs/Å,0.0095,0.0111,N/A,17.00 % ↑
7,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.5489,2.1367,N/A,16.17 % ↓
8,sepd,peak,,broad_gauss_sigma_0,μs²,3.0148,4.9491,N/A,64.16 % ↑
9,sepd,peak,,broad_gauss_sigma_1,μs/Å,33.3451,34.4752,N/A,3.39 % ↑
10,sepd,instrument,,d_to_tof_offset,μs,-8.7248,-8.7247,N/A,0.00 % ↓


#### Display Correlations

In [48]:
project.display.fit.correlations()

⚠️ Correlation matrix is unavailable for this fit. Use a minimizer that returns covariance information or posterior samples.      


#### Display Pattern

In [49]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

In [50]:
project.display.pattern(expt_name='sepd', x='d_spacing')

## 💾 Save Project

In [51]:
project.save_as(dir_path='projects/refine-si-sepd')

Saving project 📦 'si_sepd' to '../../../projects/refine-si-sepd'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 si.edi


├── 📁 experiments/


│   └── 📄 sepd.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 si_sepd.html
